## <code>static Word Embedding</code>

###  Skip-gram and CBOW Word Embedding Trainer

###  Purpose

This notebook defines the `SkipgramTrainer`, `CBOWTrainer` class used to train a **Skip-gram Word2Vec model** and **CBOW Word2Vec model** respectively from a collection of raw text documents (e.g., AP or WSJ datasets).

---

###  What This Code Does

1. **Preprocess Text Data**
   - Removes HTML tags, punctuation, and stopwords
   - Converts text to lowercase and tokenizes it
   - Filters out non-alphabetic tokens

2. **Prepare Training Data**
   - Iterates over all documents
   - Converts them into a list of tokenized word lists suitable for Word2Vec training

3. **Train Skip-gram Model**
   - Uses the `gensim` library to train a Word2Vec model with the Skip-gram architecture (`sg=1`) and (`sg=0`) for CBOW 
   - Configurable parameters:
     - `vector_size`: Dimensionality of word vectors (default: 300)
     - `window`: Context window size (default: 5)
     - `min_count`: Minimum frequency to consider a word (default: 5)
     - `negative`: Number of negative samples (default: 15)
     - `epochs`: Number of training iterations (default: 50)

4. **Save the Model**
   - The trained model is saved in the Word2Vec binary format for reuse in other components (like WordTranslator)

5. **Test Word Translation**
   - Loads the saved embeddings using a WordTranslator
   - Outputs translation candidates for a few example words using cosine similarity


In [ ]:
import pandas as pd
import numpy as np
import string
import re
import math
import os
import gensim
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from typing import Dict, List, Tuple
from collections import defaultdict

### Word Translation for Testing 

In [ ]:
class WordTranslator:
    def __init__(self, word_embeddings_path: str):
        """
        Initialize the word translator with pre-trained word embeddings
        """
        self.word_embeddings = gensim.models.KeyedVectors.load_word2vec_format(
            word_embeddings_path, binary=True
        )
        self.vocabulary = set(self.word_embeddings.index_to_key)
        
    def get_nearest_neighbors(self, word: str, k: int = 8) -> List[Tuple[str, float]]:
        """
        Find k nearest neighbors for a given word based on cosine similarity
        """
        if word not in self.vocabulary:
            return []
        
        similar_words = self.word_embeddings.most_similar(word, topn=k)
        return similar_words
    
    def calculate_translation_probability(self, source_word: str, target_word: str,
                                       temperature: float = 0.1) -> float:
        """
        Calculate translation probability pt(w|u) using NTLM approach
        Args:
            source_word: The source word u
            target_word: The target word w
            temperature: Temperature parameter for controlling probability distribution
        """
        if source_word not in self.vocabulary or target_word not in self.vocabulary:
            return 0.0
        
        # Calculate cosine similarity
        similarity = self.word_embeddings.similarity(source_word, target_word)
        
        # Convert similarity to probability using softmax-like normalization
        probability = math.exp(similarity / temperature)
        
        # Get normalization term (sum of exp(sim(u,w')/temperature) for all w' in vocabulary)
        # Note: For efficiency, we only consider top-k nearest neighbors
        nearest_neighbors = self.get_nearest_neighbors(source_word, k=8)
        normalization_term = sum(math.exp(sim / temperature) 
                               for _, sim in nearest_neighbors)
        
        return probability / normalization_term if normalization_term > 0 else 0.0
    
    def get_translation_candidates(self, source_word: str, 
                                 threshold: float = 0.01) -> List[Tuple[str, float]]:
        """
        Get all possible translation candidates with their probabilities
        Args:
            source_word: The source word to translate
            threshold: Minimum probability threshold for considering a translation
        """
        if source_word not in self.vocabulary:
            return []
        
        # Get nearest neighbors as potential translation candidates
        candidates = self.get_nearest_neighbors(source_word, k=8)
        
        # Calculate translation probabilities for each candidate
        translation_probs = []
        for candidate, _ in candidates:
            prob = self.calculate_translation_probability(source_word, candidate)
            if prob >= threshold:
                translation_probs.append((candidate, prob))
        
        # Sort by probability in descending order
        return sorted(translation_probs, key=lambda x: x[1], reverse=True)

### Preprocessing for datasets (AP or WSJ)

In [6]:
# AP Data
# Function to parse the TREC file
def parse_trec_file(trec_file_path):
    doc_texts = {}
    current_doc_id = None
    current_text = []
    
    encodings = ['utf-8', 'latin-1', 'ISO-8859-1']
    for encoding in encodings:
        try:
            with open(trec_file_path, 'r', encoding=encoding, errors='ignore') as file:
                for line in file:
                    if line.startswith('<DOCNO>'):
                        current_doc_id = line.strip().replace('<DOCNO>', '').replace('</DOCNO>', '').strip()
                    elif line.startswith('</TEXT>'):
                        if current_doc_id:
                            doc_texts[current_doc_id] = ' '.join(current_text)
                            current_doc_id = None
                            current_text = []
                    elif current_doc_id:
                        if not (line.startswith('<DOC>') or line.startswith('</DOC>') or line.startswith('<FILEID>') or
                                line.startswith('<FIRST>') or line.startswith('<SECOND>') or line.startswith('<HEAD>') or
                                line.startswith('</BYLINE>') or
                                line.startswith('<DATELINE>') or line.startswith('<TEXT>')):
                            current_text.append(line.strip())
            break
        except UnicodeDecodeError:
            continue  

    return doc_texts

# Path to your concatenated TREC file
trec_file_path = os.path.join("..", "Data", "AP_Doc", "ap", "concatenated", "concatenated_documents.txt")
#trec_file_path = os.path.join("..", "Data", "WSJ_DOC", "wsj", "concatenated_WSJ", "concatenated_WSJ.txt")

# Parse the document texts
doc_texts = parse_trec_file(trec_file_path)



### Skipgram Model For AP and WSJ Datasets

In [ ]:
class SkipgramTrainer:
    def __init__(self):
        """Initialize the trainer with necessary NLTK resources"""
        nltk.download('punkt')
        nltk.download('stopwords')
        self.stop_words = set(stopwords.words('english'))
        
    def preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text by removing special characters, converting to lowercase,
        removing stopwords and tokenizing
        """
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords and non-alphabetic tokens
        tokens = [token for token in tokens 
                 if token not in self.stop_words 
                 and token.isalpha()]
        
        return tokens

    def prepare_training_data(self, doc_texts: Dict[str, str]) -> List[List[str]]:
        """Convert document dictionary into format suitable for Word2Vec training"""
        training_data = []
        
        for doc_id, text in doc_texts.items():
            tokens = self.preprocess_text(text)
            if tokens:  # Only add if document contains valid tokens
                training_data.append(tokens)
                
        return training_data

    def train_skipgram_model(self, 
                            training_data: List[List[str]], 
                            vector_size: int = 300,
                            window: int = 5,
                            min_count: int = 5,
                            workers: int = 4,
                            negative: int = 15,
                            epochs: int = 50) -> Word2Vec:
        """
        Train Skip-gram model using preprocessed data
        
        Args:
            training_data: List of tokenized documents
            vector_size: Dimensionality of word vectors
            window: Maximum distance between current and predicted word
            min_count: Minimum frequency of words to consider
            workers: Number of CPU cores to use
            epochs: Number of training epochs
        """
        model = Word2Vec(sentences=training_data,
                        vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        workers=workers,
                        sg=1,  # Skip-gram model (sg=1)
                        negative =negative,
                        epochs=epochs)
        
        return model

    def save_model(self, model: Word2Vec, save_path: str):
        """Save the trained model in word2vec binary format"""
        model.wv.save_word2vec_format(save_path, binary=True)

def main():
    # Initialize trainer
    trainer = SkipgramTrainer()
    
    # Parse TREC file (using your existing parse_trec_file function)
    trec_file_path = os.path.join("..", "Data", "AP_Doc", "ap", "concatenated", "concatenated_documents.txt")
    #trec_file_path = os.path.join("..", "Data", "WSJ_DOC", "wsj", "concatenated_WSJ", "concatenated_WSJ.txt")

    doc_texts = parse_trec_file(trec_file_path)
    
    # Prepare training data
    print("Preparing training data...")
    training_data = trainer.prepare_training_data(doc_texts)
    
    # Train model
    print("Training Skip-gram model...")
    model = trainer.train_skipgram_model(training_data)
    
    # Save model
    save_path = os.path.join("..", "Data", "Word_Embedding", "AP_skipgram_model35.bin")
    print(f"Saving model to {save_path}...")
    trainer.save_model(model, save_path)
    
    # Test the model with your existing WordTranslator
    print("\nTesting translation capabilities...")
    translator = WordTranslator(save_path)
    
    test_words = ["oil", "gas", "energy", "company"]
    for word in test_words:
        print(f"\nTranslation candidates for '{word}':")
        translations = translator.get_translation_candidates(word)
        for target_word, prob in translations:
            print(f"{target_word}: {prob:.4f}")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dolla\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dolla\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preparing training data...
Training Skip-gram model...
Saving model to ..\Data\Word_Embedding\AP_skipgram_model35.bin...

Testing translation capabilities...

Translation candidates for 'oil':
crude: 0.5012
petroleum: 0.1484
barrels: 0.0753
spill: 0.0579
barrel: 0.0571
heating: 0.0557
refineries: 0.0527
opec: 0.0518

Translation candidates for 'gas':
tear: 0.2549
natural: 0.1652
canisters: 0.1222
gasoline: 0.0980
isocynate: 0.0955
propanebutane: 0.0918
nonproducing: 0.0873
methane: 0.0851

Translation candidates for 'energy':
ahearne: 0.1660
nonfossil: 0.1556
energys: 0.1388
fuels: 0.1164
commerce: 0.1111
redoglio: 0.1101
watkins: 0.1037
tokamak: 0.0983

Translation candidates for 'company':
companys: 0.3569
companies: 0.1554
subsidiary: 0.1493
subsidiaries: 0.0823
maker: 0.0702
manufacturer: 0.0648
corporation: 0.0629
anac: 0.0581


### CBOW Model For AP and WSJ Datasets

In [ ]:
class CBOWTrainer:
    def __init__(self):
        """Initialize the trainer with necessary NLTK resources"""
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        self.stop_words = set(stopwords.words('english'))
        
    def preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text by removing special characters, converting to lowercase,
        removing stopwords and tokenizing
        """
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords, non-alphabetic tokens, and very short words
        tokens = [token for token in tokens 
                 if token not in self.stop_words 
                 and token.isalpha()
                 and len(token) > 2]  # Remove very short words
        
        return tokens

    def prepare_training_data(self, doc_texts: Dict[str, str]) -> List[List[str]]:
        """Convert document dictionary into format suitable for Word2Vec training"""
        training_data = []
        total_tokens = 0
        
        for doc_id, text in doc_texts.items():
            tokens = self.preprocess_text(text)
            if tokens:
                training_data.append(tokens)
                total_tokens += len(tokens)
        
        #print(f"Prepared {len(training_data)} documents with {total_tokens} total tokens")
        return training_data

    def train_cbow_model(self, 
                        training_data: List[List[str]], 
                        vector_size: int = 300,
                        window: int = 5,
                        min_count: int = 5,
                        workers: int = 4,
                        epochs: int = 30,
                        negative: int = 15,
                        alpha: float = 0.025,
                        min_alpha: float = 0.0001) -> Word2Vec:
        """
        Train CBOW model using preprocessed data with improved parameters
        
        Args:
            training_data: List of tokenized documents
            vector_size: Dimensionality of word vectors
            window: Maximum distance between current and predicted word
            min_count: Minimum frequency of words to consider
            workers: Number of CPU cores to use
            epochs: Number of training epochs
            negative: Number of negative samples
            alpha: Initial learning rate
            min_alpha: Minimum learning rate
        """
        # Calculate dynamic learning rate decay
        alpha_delta = (alpha - min_alpha) / epochs
        
        # Initialize model with improved parameters
        model = Word2Vec(vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        workers=workers,
                        sg=0,  # CBOW model
                        negative=negative,
                        alpha=alpha,
                        min_alpha=min_alpha,
                        compute_loss=True)
        
        # Build vocabulary
        model.build_vocab(training_data)
        
        # Train the model with progress monitoring
        total_examples = len(training_data)
        
        losses = []
        for epoch in range(epochs):
            current_alpha = alpha - (alpha_delta * epoch)
            model.alpha = current_alpha
            model.min_alpha = current_alpha
            
            model.train(training_data,
                       total_examples=total_examples,
                       epochs=1,
                       compute_loss=True)
            
            current_loss = model.get_latest_training_loss()
            losses.append(current_loss)
        
        return model

    def save_model(self, model: Word2Vec, save_path: str):
        """Save the trained model in word2vec binary format"""
        model.wv.save_word2vec_format(save_path, binary=True)
        
    
    def evaluate_model(self, model: Word2Vec, test_words: List[str]):
        """
        Evaluate the model by printing similar words and their similarities
        for a list of test words
        """
        print("\nModel Evaluation:")
        for word in test_words:
            try:
                similar_words = model.wv.most_similar(word, topn=5)
                print(f"\nSimilar words to '{word}':")
                for similar_word, similarity in similar_words:
                    print(f"  {similar_word}: {similarity:.4f}")
            except KeyError:
                print(f"\nWord '{word}' not in vocabulary")


def main():
    # Initialize trainer
    trainer = CBOWTrainer()
    
    # Parse TREC file (using your existing parse_trec_file function)
    #trec_file_path = os.path.join("..", "Data", "AP_Doc", "ap", "concatenated", "concatenated_documents.txt")
    trec_file_path = os.path.join("..", "Data", "WSJ_DOC", "wsj", "concatenated_WSJ", "concatenated_WSJ.txt")

    doc_texts = parse_trec_file(trec_file_path)
    
    # Prepare training data
    print("Preparing training data...")
    training_data = trainer.prepare_training_data(doc_texts)
    
    # Train model with improved parameters
    print("Training CBOW model...")
    model = trainer.train_cbow_model(
        training_data,
        vector_size=300,
        window=5,
        min_count=5,
        workers=4,
        epochs=30,
        negative=15,
        alpha=0.025,
        min_alpha=0.0001
    )
    
    # Save model
    save_path = os.path.join("..", "Data", "Word_Embedding", "WSJ_cbow_model.bin")
    print(f"Saving model to {save_path}...")
    trainer.save_model(model, save_path)
    
    # Evaluate model
    test_words = ["oil", "gas", "energy", "company", "pneumonia"]
    trainer.evaluate_model(model, test_words)

if __name__ == "__main__":
    main()

Preparing training data...
Training CBOW model...
Saving model to ..\Data\Word_Embedding\WSJ_cbow_model.bin...

Model Evaluation:

Similar words to 'oil':
  crudeoil: 0.5901
  petroleum: 0.5393
  wellhead: 0.5278
  oils: 0.5068
  naturalgas: 0.4861

Similar words to 'gas':
  naturalgas: 0.7154
  gass: 0.4934
  pipeline: 0.4856
  wainoco: 0.4742
  gasrelated: 0.4730

Similar words to 'energy':
  energys: 0.5834
  transportation: 0.4638
  oil: 0.4558
  naturalgas: 0.4370
  gas: 0.4284

Similar words to 'company':
  companys: 0.7241
  concern: 0.6423
  companies: 0.5624
  maker: 0.4718
  retailer: 0.4680

Word 'pneumoni' not in vocabulary
